# 🔬 EXODUS-SPECULUM - Génération Golden Samples

Ce notebook génère les échantillons de référence (golden samples) pour les tests.

**Phase 2.5B: Golden Tests Framework**

---

## 1. Setup Environment

In [ ]:
# EXODUS-SPECULUM - Génération Golden Samples
# Ce notebook génère les échantillons de référence pour les tests

!git clone https://github.com/kioka8877-ux/-EXODUS-SPECULUM-.git 2>/dev/null || echo "Repo exists"
%cd -EXODUS-SPECULUM-
!git pull
!pip install -q numpy opencv-python-headless Pillow

## 2. Generate Mock Frame

In [ ]:
import numpy as np
import cv2
from pathlib import Path

# Créer frame de test
def create_test_frame(width=960, height=540):
    """Génère une frame RGB simulant un intérieur."""
    frame = np.zeros((height, width, 3), dtype=np.uint8)
    # Gradient simulant un intérieur
    for y in range(height):
        for x in range(width):
            frame[y, x] = [
                int(100 + 50 * (x / width)),    # R - gradient horizontal
                int(120 + 60 * (y / height)),   # G - gradient vertical
                140                              # B - constant
            ]
    return frame

frame = create_test_frame()
Path("tests/golden/input").mkdir(parents=True, exist_ok=True)
cv2.imwrite("tests/golden/input/test_frame_001.png", frame)
print(f"✅ Frame créée: {frame.shape}")
print(f"   Dtype: {frame.dtype}")
print(f"   Range: [{frame.min()}, {frame.max()}]")

## 3. Generate Mock Depth Map

In [ ]:
# Créer depth map de test (16-bit)
def create_test_depth(width=960, height=540):
    """Génère une depth map 16-bit simulant une pièce."""
    y = np.linspace(0, 1, height)[:, np.newaxis]
    x = np.linspace(0, 1, width)[np.newaxis, :]
    
    # Profondeur de base (plus proche en bas, plus loin en haut)
    depth = (1 - y * 0.7) * 50000 + 10000
    
    # Variation horizontale
    depth += x * 2000 - 1000
    
    # Ajout de bruit réaliste
    depth += np.random.normal(0, 500, (height, width))
    
    return np.clip(depth, 0, 65535).astype(np.uint16)

depth = create_test_depth()
Path("tests/golden/f01_depth").mkdir(parents=True, exist_ok=True)
np.savez_compressed("tests/golden/f01_depth/test_depth_001.npz", depth=depth)

print(f"✅ Depth créée: {depth.shape}")
print(f"   Dtype: {depth.dtype}")
print(f"   Range: [{depth.min()}, {depth.max()}]")
print(f"   Mean: {depth.mean():.1f}")
print(f"   Std: {depth.std():.1f}")

## 4. Generate Multiple Samples (Batch)

In [ ]:
# Générer un batch de samples
BATCH_SIZE = 5

print(f"🔄 Generating {BATCH_SIZE} frame/depth pairs...")

for i in range(BATCH_SIZE):
    # Variation des paramètres
    room_color = (
        100 + np.random.randint(-20, 20),
        120 + np.random.randint(-20, 20),
        140 + np.random.randint(-20, 20)
    )
    
    # Frame avec couleur variée
    frame = create_test_frame()
    frame = np.clip(frame.astype(int) + np.random.randint(-10, 10, frame.shape), 0, 255).astype(np.uint8)
    cv2.imwrite(f"tests/golden/input/test_frame_{i+1:03d}.png", frame)
    
    # Depth avec variation
    depth = create_test_depth()
    np.savez_compressed(f"tests/golden/f01_depth/test_depth_{i+1:03d}.npz", depth=depth)
    
    print(f"   [{i+1}/{BATCH_SIZE}] Generated pair {i+1:03d}")

print(f"\n✅ Batch generation complete!")

## 5. Validate Generated Samples

In [ ]:
# Validation des samples générés
import sys
sys.path.insert(0, '.')

from tests.validators.depth_validator import DepthValidator
from pathlib import Path

validator = DepthValidator("tests/contracts/depth_contract.json")

depth_files = list(Path("tests/golden/f01_depth").glob("*.npz"))
print(f"🔍 Validating {len(depth_files)} depth files...\n")

passed = 0
failed = 0

for f in depth_files:
    result = validator.validate(str(f))
    if result.valid:
        passed += 1
        print(f"✅ {f.name}: VALID")
    else:
        failed += 1
        print(f"❌ {f.name}: INVALID - {result.errors}")

print(f"\n📊 Results: {passed} passed, {failed} failed")

## 6. Commit Golden Samples

In [ ]:
# Afficher statut Git
!git add tests/golden/
!git status

print("\n" + "="*60)
print("⚠️  Pour commit, exécutez manuellement:")
print('!git commit -m "TST: Add golden samples"')
print('!git push')
print("="*60)